In [1]:
%pip install statsmodels
%pip install pandas
%pip install numpy 
%pip install python-math
%pip install networkx

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd 
import numpy as np
import statsmodels.api as sm
import math


In [3]:
data = pd.read_csv("/Users/parkn/Downloads/DGS1 quarterly observations 20 years.csv")

data['observation_date1'] = pd.to_datetime(data['observation_date'])

data = data.drop(columns=['observation_date'])  

data = data.set_index('observation_date1')

data_yearly = data.resample('Y').first()  

data = data_yearly.reset_index()

data


FileNotFoundError: [Errno 2] No such file or directory: '/Users/parkn/Downloads/DGS1 quarterly observations 20 years.csv'

In [ ]:
face_value = 100     
coupon_rate = 0.035  
maturity = 6         
frequency = 1        
coupon = face_value * coupon_rate / frequency

In [ ]:
rate_0_1 = 3.880 / 100
rate_0_2 = 3.985 / 100
rate_0_3 = 4.085 / 100
rate_0_4 = 4.163 / 100
rate_0_5 = 4.284 / 100
rate_0_6 = 4.385 / 100

coupon_1 = coupon * np.exp(-rate_0_1 * 1)
coupon_2 = coupon * np.exp(-rate_0_2 * 2)
coupon_3 = coupon * np.exp(-rate_0_3 * 3)
coupon_4 = coupon * np.exp(-rate_0_4 * 4)
coupon_5 = coupon * np.exp(-rate_0_5 * 5)
coupon_6 = coupon * np.exp(-rate_0_6 * 6) + face_value * np.exp(-rate_0_6 * 6)

price_strips = coupon_1 + coupon_2 + coupon_3 + coupon_4 + coupon_5 + coupon_6
print('price using STRIPS curve: $', price_strips)

delta_noncallable = (coupon_1 * 1 + coupon_2 * 2 + coupon_3 * 3 + coupon_4 * 4 + coupon_5 * 5 + coupon_6 * 6) / 100
print('Delta noncallable: $', delta_noncallable)



In [ ]:
from statsmodels.tsa.ar_model import AutoReg

ar_model = AutoReg(data['DGS1'], lags=1).fit()

print(ar_model.summary())

long_term_mean = ar_model.params[0] / (1 - ar_model.params[1])
mean_reversion_speed = ar_model.params[1]
volatility = np.sqrt(ar_model.sigma2)
print(f"Long-term mean: {long_term_mean}")
print(f"Mean reversion speed: {mean_reversion_speed}")
print(f"Volatility: {volatility}")

In [ ]:
phi = mean_reversion_speed       
mu = long_term_mean               
sigma = volatility              
dt = 1   

step_1 = volatility * np.sqrt(-2 * np.log(mean_reversion_speed)) * np.sqrt(1)
step_1 = round(step_1, 3)
step_1 = .115

rate_0_1 = 3.88

In [ ]:
def build_rate_tree(r0, step, n_periods):
    tree = [[r0]]

    for t in range(1, n_periods + 1):
        prev = tree[-1]
        row = [(r + step) for r in prev] + [(prev[-1] - step)]
        tree.append(row)

    tree = [[round(r, 4) for r in row] for row in tree]
    return tree


In [ ]:
import matplotlib.pyplot as plt

rate_tree = build_rate_tree(rate_0_1, step_1, maturity)

plt.figure(figsize=(10, 6))
for i, level in enumerate(rate_tree):
    x = [i] * len(level) 
    y = level 
    plt.scatter(x, y, label=f"Level {i}")
    for j, rate in enumerate(level):
        plt.text(x[j], y[j], f"{rate:.3f}", fontsize=12, ha='right', va='bottom')  

plt.title("Rate Tree Visualization")
plt.xlabel("Time Period")
plt.ylabel("Interest Rate")
plt.legend()
plt.grid()
plt.show()
print("Rate tree: ")
rate_tree


In [ ]:
def compute_dynamic_p(r, phi, mu, dt, sigma):
    p = (1 / 2) + ((mu - r) * np.sqrt(-np.log(phi)) / (sigma * np.sqrt(8)))
    return p



In [ ]:
def price_coupon_bond_vasicek(rate_tree, coupon, face_value, dt, phi, mu, step, sigma): 
    n = len(rate_tree) - 1
    price_tree = [[0] * len(level) for level in rate_tree]  

    price_tree[-1] = [face_value + coupon for _ in rate_tree[-1]]

    for t in range(n - 1, -1, -1):
        for i in range(t + 1):
            r = rate_tree[t][i]
            p = compute_dynamic_p(r, phi, mu, dt, sigma)
            expected = ((p) * price_tree[t + 1][i]) + ((1-p) * price_tree[t + 1][i + 1])
            discounted = np.exp(-r * .01 * dt) * expected
            price_tree[t][i] = float(round(coupon + discounted, 3))
    return price_tree

In [ ]:
vasicek_price_tree = price_coupon_bond_vasicek(rate_tree, coupon, face_value, dt, phi, mu, step_1, sigma)
print ('noncallable bond price using Vasicek model: $', vasicek_price_tree[0][0])

delta_noncallable = - (vasicek_price_tree[1][0] - vasicek_price_tree[1][1]) / (rate_tree[1][0] - rate_tree[1][1])
print('Delta noncallable bond using Vasicek model: $', delta_noncallable)

def rate_shift(rate_tree, shift):
    shifted_tree = []
    for level in rate_tree:
        shifted_level = [r + shift for r in level]
        shifted_tree.append(shifted_level)
    return shifted_tree
shifted_tree = rate_shift(rate_tree, 0.001)
price_shifted = price_coupon_bond_vasicek(shifted_tree, coupon, face_value, dt, phi, mu, step_1, sigma)[0][0]

delta_noncallable_shift = -(price_shifted - vasicek_price_tree[0][0]) / 0.001
print('Delta noncallable bond using shift method: $', delta_noncallable_shift)

In [ ]:
plt.figure(figsize=(10, 6))
for i, level in enumerate(vasicek_price_tree):
    x = [i] * len(level) 
    y = level  
    plt.scatter(x, y, label=f"Level {i}")
    for j, rate in enumerate(level):
        plt.text(x[j], y[j], f"{rate:.3f}", fontsize=12, ha='right', va='bottom')  
plt.title("Price Tree Visualization")
plt.xlabel("Time Period")
plt.ylabel("Price")
plt.legend()
plt.grid()
plt.show()

In [ ]:
def build_probability_tree(rate_tree, phi, mu, dt, sigma):
    n = len(rate_tree)
    prob_tree = [[0.0] * len(row) for row in rate_tree]
    prob_tree[0][0] = 1.0 

    for t in range(n - 1):  
        for i in range(len(rate_tree[t])):  
            r = rate_tree[t][i]
            p = compute_dynamic_p(r, phi, mu, dt, sigma)
            
            prob_tree[t + 1][i]     += prob_tree[t][i] * p
            prob_tree[t + 1][i + 1] += prob_tree[t][i] * (1 - p)

    prob_tree = [[round(float(prob), 3) for prob in row] for row in prob_tree]
    return prob_tree
probability_tree = build_probability_tree(rate_tree, phi, mu, dt, sigma)
probability_tree

In [ ]:
exercise_price = 103

t_expire = 5

def european_call_option_value(price_tree, exercise_price, prob_tree, t_expire):
    total = 0.0
    for i in range(len(price_tree[t_expire])):
        payoff = max(0, price_tree[t_expire][i] - exercise_price)
        weighted_payoff = prob_tree[t_expire][i] * payoff
        total += weighted_payoff
    return float(total)

prob_tree = build_probability_tree(rate_tree, phi, mu, dt, sigma)

call_option_price = european_call_option_value(vasicek_price_tree, exercise_price, prob_tree, t_expire)
print("European call option price:", call_option_price)

callable_bond_price = vasicek_price_tree[0][0] - call_option_price
print("European Callable bond price: ", callable_bond_price)


def price_european_call_option_tree(price_tree, exercise_price, prob_tree, dt, rate_tree):
    option_tree = [[0] * len(level) for level in price_tree]
    
    T = len(price_tree) - 1  

    for i in range(len(price_tree[T])):
        option_tree[T][i] = max(price_tree[T][i] - exercise_price, 0)

    for t in range(T - 1, -1, -1):
        for i in range(len(price_tree[t])):
            discount_factor = np.exp(-rate_tree[t][i] * 0.01 * dt)
            expected_continuation = (
                prob_tree[t + 1][i] * option_tree[t + 1][i] +
                prob_tree[t + 1][i + 1] * option_tree[t + 1][i + 1]
            )

            option_tree[t][i] = expected_continuation * discount_factor

    return option_tree
european_option_tree = price_european_call_option_tree(
    vasicek_price_tree,
    exercise_price,
    prob_tree,
    dt,
    rate_tree
)

delta_euro_option = - (european_option_tree[1][0] - european_option_tree[1][1]) / (rate_tree[1][0] - rate_tree[1][1])
delta_european = delta_noncallable - delta_euro_option
print("Delta of European: ", delta_european)


In [ ]:
def am_call_option(price_tree, exercise_price, prob_tree, rate_tree):
    option_tree = [[0] * len(level) for level in price_tree]

    T = len(price_tree) - 1
    for i in range(len(price_tree[T])):
        option_tree[T][i] = max(price_tree[T][i] - exercise_price, 0)

    for t in range(T - 1, -1, -1):  
        for i in range(len(price_tree[t])):
            expected_continuation = (
                prob_tree[t + 1][i] * option_tree[t + 1][i] +
                prob_tree[t + 1][i + 1] * option_tree[t + 1][i + 1]
            ) * np.exp(-rate_tree[t][i] * 0.01 * dt)

            immediate_exercise = max(price_tree[t][i] - exercise_price, 0)

            option_tree[t][i] = max(immediate_exercise, expected_continuation)

    return option_tree

option_tree = am_call_option(vasicek_price_tree, exercise_price, prob_tree, rate_tree)
american_call_option_price = option_tree[0][0]
print("American call option price:", american_call_option_price)

american_callable_bond_price = vasicek_price_tree[0][0] - american_call_option_price
print("American Callable bond price:", american_callable_bond_price)

delta_am_option = - (option_tree[1][0] - option_tree[1][1]) / (rate_tree[1][0] - rate_tree[1][1])
delta_american = delta_noncallable - delta_am_option
print("Delta of American: ", delta_american)
